# 4AAA — Multi-Temperature Unified Optuna TPE + MLflow Optimization

This notebook builds on **4AA** but replaces the custom Gaussian-process search
with **Optuna's multivariate Tree-structured Parzen Estimator (TPE)**.

One shared ESN–XGBoost hyperparameter configuration is optimized for the pooled
30–60°C dataset. Every Optuna candidate is evaluated across all 14 material-held-
out LOMO folds using exactly the same candidate parameters. The 336 out-of-fold
predictions are pooled to calculate candidate performance.

Candidate selection minimizes:

\[
J = \mathrm{pooled\ NRMSE}_{\mathrm{range}}
    + 0.25\,\mathrm{SD}(\mathrm{per	ext{-}material\ fold\ NRMSE})
\]

The study uses **100 trials**, including **20 startup observations** before TPE
relies primarily on its learned density models. Two sensible anchor configurations
are evaluated first.

MLflow records the complete optimization as one parent run and each Optuna trial
as a child run. The Optuna study is intentionally kept in memory—there is no Optuna
persistence—so an interrupted TPE study cannot resume its sampler state automatically.
MLflow uses its own local SQLite tracking backend because current MLflow releases
have deprecated the older folder-only tracking backend.

The final JSON uses the same ESN/XGBoost parameter schema as 4AA for downstream
fixed-parameter analysis.


# 1. Imports and experiment configuration


In [1]:
from datetime import datetime, timezone
from pathlib import Path
import json
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy import linalg
from scipy.signal import savgol_filter
from scipy.stats import kurtosis, skew
from sklearn.compose import TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

try:
    import mlflow
    import optuna
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Notebook 4AAA requires Optuna and MLflow. Install them in the active "
        "notebook kernel with `%pip install optuna mlflow`, then restart the kernel."
    ) from exc

optuna.logging.set_verbosity(optuna.logging.WARNING)


/Users/markkeanujamesexconde/.virtualenvs/warmth-sensor-object-detection/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
TEMPERATURE_FOLDERS = ("30C", "40C", "50C", "60C")
OPTIMIZATION_TARGET = "eff"
RANDOM_STATE = 42
RESERVOIR_SEEDS = (42, 43, 44)

ANALYSIS_WINDOW = (0.0, 5.0)
EARLY_WINDOW = (0.0, 1.0)
MID_WINDOW = (1.0, 3.0)
LATE_WINDOW = (3.0, 5.0)

# Defaults are used only if a function is called without an Optuna candidate.
ESN_RES_SIZE = 20
ESN_LEAK_RATE = 0.1
ESN_INPUT_MAGNITUDE = 1.0
ESN_SPECTRAL_RADIUS = 0.9
ESN_WASHOUT = 0

# Baseline values are overridden by every joint Optuna candidate.
XGB_PARAMS = {
    "n_estimators": 300, "max_depth": 3, "learning_rate": 0.03,
    "subsample": 0.85, "colsample_bytree": 0.85,
    "min_child_weight": 2.0, "reg_alpha": 0.0, "reg_lambda": 1.0,
}

OPTUNA_N_TRIALS = 100
OPTUNA_N_STARTUP_TRIALS = 20
OPTUNA_MULTIVARIATE_TPE = True
OPTUNA_SEED = 4042
STABILITY_WEIGHT = 0.25
UNIFIED_CV_PROTOCOL = "LeaveOneGroupOut by material"

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in (cwd, cwd.parent) if (path / "data").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from the project root or notebooks directory.")
DATA_ROOT = PROJECT_ROOT / "data" / "02_preprocessed"
missing_folders = [name for name in TEMPERATURE_FOLDERS if not (DATA_ROOT / name).is_dir()]
if missing_folders:
    raise FileNotFoundError(f"Missing temperature data folders: {missing_folders}")

RESULTS_DIR = PROJECT_ROOT / "results" / "multi_temp_esn"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
temperature_tag = "-".join(TEMPERATURE_FOLDERS)
RESULT_STEM = f"{temperature_tag}_{OPTIMIZATION_TARGET}_v7_optuna_tpe_mlflow"
PARAMETER_FILE = RESULTS_DIR / f"{RESULT_STEM}_parameters.json"
TRIAL_CHECKPOINT_FILE = RESULTS_DIR / f"{RESULT_STEM}_trial_checkpoint.csv"

# Local MLflow tracking; this SQLite file belongs to MLflow, not to Optuna.
# Optuna itself remains an in-memory study and cannot resume after interruption.
MLFLOW_DB_FILE = RESULTS_DIR / "4AAA_mlflow_tracking.db"
MLFLOW_ARTIFACT_DIR = RESULTS_DIR / "mlflow_artifacts"
MLFLOW_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MLFLOW_TRACKING_URI = f"sqlite:///{MLFLOW_DB_FILE.resolve()}"
MLFLOW_EXPERIMENT_NAME = "4AAA_multi_temp_unified_optuna_tpe"
MLFLOW_RUN_NAME = datetime.now(timezone.utc).strftime("4AAA_%Y%m%dT%H%M%SZ")

print("Data root:", DATA_ROOT)
print("Temperature folders:", TEMPERATURE_FOLDERS)
print("Optuna trials:", OPTUNA_N_TRIALS)
print("MLflow tracking URI:", MLFLOW_TRACKING_URI)
print("Final parameter file:", PARAMETER_FILE)


Data root: /Users/markkeanujamesexconde/Library/CloudStorage/OneDrive-Personal/Documents/Academe Files/4. Doctoral/Research 1 - Warmth Sensor/Warmth-Sensor-Object-Detection/data/02_preprocessed
Temperature folders: ('30C', '40C', '50C', '60C')
Optuna trials: 100
MLflow tracking URI: sqlite:////Users/markkeanujamesexconde/Library/CloudStorage/OneDrive-Personal/Documents/Academe Files/4. Doctoral/Research 1 - Warmth Sensor/Warmth-Sensor-Object-Detection/results/multi_temp_esn/4AAA_mlflow_tracking.db
Final parameter file: /Users/markkeanujamesexconde/Library/CloudStorage/OneDrive-Personal/Documents/Academe Files/4. Doctoral/Research 1 - Warmth Sensor/Warmth-Sensor-Object-Detection/results/multi_temp_esn/30C-40C-50C-60C_eff_v7_optuna_tpe_mlflow_parameters.json


# 2. Standard material properties and trial loading

The processed CSV values in `k`, `Mass`, `Volume`, `rho`, and `cp` are deliberately
ignored. After loading the sensor data, the notebook overwrites those fields using
the authoritative table below.

`Mass=1 kg` and `Volume=1 m³` are placeholders and can be updated later.


In [3]:
STANDARD_PROPERTIES = {
    "ps_foam":   {"k": 0.034, "Mass": 1.0, "Volume": 1.0, "rho": 25.0,   "cp": 1400.0},
    "pu_foam":   {"k": 0.043, "Mass": 1.0, "Volume": 1.0, "rho": 30.0,   "cp": 1400.0},
    "cork":      {"k": 0.043, "Mass": 1.0, "Volume": 1.0, "rho": 240.0,  "cp": 1800.0},
    "wood":      {"k": 0.150, "Mass": 1.0, "Volume": 1.0, "rho": 700.0,  "cp": 1700.0},
    "pdms":      {"k": 0.150, "Mass": 1.0, "Volume": 1.0, "rho": 970.0,  "cp": 1460.0},
    "gypsum":    {"k": 0.170, "Mass": 1.0, "Volume": 1.0, "rho": 800.0,  "cp": 1090.0},
    "cement":    {"k": 0.290, "Mass": 1.0, "Volume": 1.0, "rho": 1440.0, "cp": 750.0},
    "graphite":  {"k": 100.0, "Mass": 1.0, "Volume": 1.0, "rho": 1820.0, "cp": 710.0},
    "bismuth":   {"k": 8.1,   "Mass": 1.0, "Volume": 1.0, "rho": 9780.0, "cp": 130.0},
    "titanium":  {"k": 21.9,  "Mass": 1.0, "Volume": 1.0, "rho": 4506.0, "cp": 523.0},
    "nickel":    {"k": 90.9,  "Mass": 1.0, "Volume": 1.0, "rho": 8908.0, "cp": 461.0},
    "iron":      {"k": 80.4,  "Mass": 1.0, "Volume": 1.0, "rho": 7874.0, "cp": 449.0},
    "aluminum":  {"k": 237.0, "Mass": 1.0, "Volume": 1.0, "rho": 2700.0, "cp": 897.0},
    "copper":    {"k": 401.0, "Mass": 1.0, "Volume": 1.0, "rho": 8960.0, "cp": 385.0},
}

SAMPLE_ALIASES = {
    "ps": "ps_foam", "ps foam": "ps_foam", "ps_foam": "ps_foam",
    "pu": "pu_foam", "pu foam": "pu_foam", "pu_foam": "pu_foam",
    "cork": "cork", "cork fine": "cork", "cork_fine": "cork",
    "wood": "wood",
    "pdms": "pdms",
    "gypsum": "gypsum",
    "cement": "cement",
    "graphite": "graphite", "carbon": "graphite",
    "bi": "bismuth", "bismuth": "bismuth",
    "ti": "titanium", "titanium": "titanium",
    "ni": "nickel", "nickel": "nickel",
    "fe": "iron", "iron": "iron",
    "al": "aluminum", "aluminum": "aluminum",
    "cu": "copper", "copper": "copper",
}

standard_table = (
    pd.DataFrame.from_dict(STANDARD_PROPERTIES, orient="index")
    .rename_axis("Material")
    .reset_index()
)
display(standard_table)

required = {"Sample", "Trial", "Time", "Primary", "Secondary"}
frames = []
load_rows = []

for temperature in TEMPERATURE_FOLDERS:
    folder = DATA_ROOT / temperature
    for path in sorted(folder.glob("*.csv")):
        frame = pd.read_csv(path)
        missing = required - set(frame.columns)
        if missing:
            warnings.warn(
                f"Skipping {temperature}/{path.name}; missing {sorted(missing)}"
            )
            continue
        frame["Temperature"] = temperature
        frame["source_file"] = path.name
        frames.append(frame)
        load_rows.append({
            "Temperature": temperature,
            "file": path.name,
            "rows_loaded": len(frame),
        })

if not frames:
    raise ValueError("No valid trial files were found.")

DATA = pd.concat(frames, ignore_index=True)
DATA = DATA.replace([np.inf, -np.inf], np.nan)
for column in ("Trial", "Time", "Primary", "Secondary"):
    DATA[column] = pd.to_numeric(DATA[column], errors="coerce")
DATA = DATA.dropna(subset=list(required) + ["Temperature"]).copy()
DATA["Trial"] = DATA["Trial"].astype(int)
DATA["Temperature_C"] = pd.to_numeric(
    DATA["Temperature"].str.extract(r"(\d+(?:\.\d+)?)", expand=False),
    errors="coerce",
)
if DATA["Temperature_C"].isna().any():
    raise ValueError("A temperature-folder name could not be converted to Celsius.")

normalized_sample = (
    DATA["Sample"].astype(str).str.strip().str.lower().str.replace("_", " ")
)
DATA["Sample"] = normalized_sample.map(SAMPLE_ALIASES)
unknown_mask = DATA["Sample"].isna()
if unknown_mask.any():
    unknown = sorted(normalized_sample[unknown_mask].unique())
    raise KeyError(f"No standard-property mapping for samples: {unknown}")

# Ignore processed-file property cells and apply one standard table everywhere.
for property_name in ("k", "Mass", "Volume", "rho", "cp"):
    DATA[property_name] = DATA["Sample"].map(
        lambda sample: STANDARD_PROPERTIES[sample][property_name]
    )

# Temperature is required in the ID because material/trial numbers repeat by folder.
DATA["trial_id"] = (
    DATA["Temperature"].astype(str)
    + "__" + DATA["Sample"].astype(str)
    + "_trial_" + DATA["Trial"].astype(str)
)
DATA["eff"] = np.sqrt(DATA["k"] * DATA["rho"] * DATA["cp"])
DATA = DATA.sort_values(["Temperature", "trial_id", "Time"]).reset_index(drop=True)

summary = (
    DATA.groupby(["trial_id", "Temperature", "Sample", "Trial"], as_index=False)
    .agg(
        n_timesteps=("Time", "size"),
        k=("k", "first"),
        eff=("eff", "first"),
    )
)
temperature_summary = (
    summary.groupby("Temperature", as_index=False)
    .agg(
        trials=("trial_id", "nunique"),
        materials=("Sample", "nunique"),
        minimum_timesteps=("n_timesteps", "min"),
        maximum_timesteps=("n_timesteps", "max"),
    )
)
print(f"Rows: {len(DATA):,}")
print(f"Unique temperature-specific trials: {DATA['trial_id'].nunique()}")
display(temperature_summary)


,Material,k,Mass,Volume,rho,cp
0,ps_foam,0.034,1.0,1.0,25.0,1400.0
1,pu_foam,0.043,1.0,1.0,30.0,1400.0
2,cork,0.043,1.0,1.0,240.0,1800.0
3,wood,0.150,1.0,1.0,700.0,1700.0
4,pdms,0.150,1.0,1.0,970.0,1460.0
5,gypsum,0.170,1.0,1.0,800.0,1090.0
6,cement,0.290,1.0,1.0,1440.0,750.0
7,graphite,100.000,1.0,1.0,1820.0,710.0
8,bismuth,8.100,1.0,1.0,9780.0,130.0
9,titanium,21.900,1.0,1.0,4506.0,523.0


Rows: 70,480
Unique temperature-specific trials: 336


,Temperature,trials,materials,minimum_timesteps,maximum_timesteps
0,30C,84,14,184,227
1,40C,84,14,200,362
2,50C,84,14,183,404
3,60C,84,14,186,326


# 3. Detect contact and align every trial

The elbow is used only to establish a common time origin. Prediction uses the fixed
0–5 second response after contact. Trials are never aligned using `k` or `eff`.


In [4]:
def find_contact_time(
    trial,
    smooth_window=15,
    polyorder=2,
    threshold_frac=0.30,
    skip_samples=5,
):
    clean = (
        trial[["Time", "Primary"]]
        .apply(pd.to_numeric, errors="coerce")
        .dropna()
        .sort_values("Time")
        .drop_duplicates("Time")
        .reset_index(drop=True)
    )
    time = clean["Time"].to_numpy(float)
    signal = clean["Primary"].to_numpy(float)
    if len(signal) < skip_samples + 7 or np.any(np.diff(time) <= 0):
        raise ValueError("Insufficient or invalid time samples.")

    work_time = time[skip_samples:]
    work_signal = signal[skip_samples:]
    window = min(int(smooth_window), len(work_signal))
    if window % 2 == 0:
        window -= 1
    minimum = polyorder + 2
    if minimum % 2 == 0:
        minimum += 1
    if window < minimum:
        raise ValueError("Sequence is too short for smoothing.")

    smooth = savgol_filter(work_signal, window, polyorder, mode="interp")
    derivative = np.gradient(smooth, work_time)
    strongest = int(np.argmin(derivative))
    active = derivative < threshold_frac * derivative[strongest]
    elbow = 0
    for position in range(strongest, -1, -1):
        if not active[position]:
            elbow = position + 1
            break
    return float(work_time[elbow])


In [5]:
aligned_trials = {}
alignment_rows = []

for trial_id, trial in DATA.groupby("trial_id", sort=False):
    trial = trial.sort_values("Time").drop_duplicates("Time").copy()
    try:
        contact_time = find_contact_time(trial)
    except ValueError as exc:
        warnings.warn(f"Skipping {trial_id}: {exc}")
        continue

    trial["time_from_contact"] = trial["Time"] - contact_time
    start, end = ANALYSIS_WINDOW
    trial = trial[
        trial["time_from_contact"].between(start, end, inclusive="both")
    ].copy()
    if len(trial) < 10:
        warnings.warn(f"Skipping {trial_id}: too few post-contact samples")
        continue
    aligned_trials[trial_id] = trial.reset_index(drop=True)
    alignment_rows.append({
        "trial_id": trial_id,
        "Temperature": trial["Temperature"].iloc[0],
        "Temperature_C": float(trial["Temperature_C"].iloc[0]),
        "Sample": trial["Sample"].iloc[0],
        "Trial": int(trial["Trial"].iloc[0]),
        "contact_time": contact_time,
        "n_analysis_samples": len(trial),
    })

ALIGNMENT = pd.DataFrame(alignment_rows)
print(f"Aligned trials retained: {len(aligned_trials)}")
display(ALIGNMENT.head())


Aligned trials retained: 336


,trial_id,Temperature,Temperature_C,Sample,Trial,contact_time,n_analysis_samples
0,30C__aluminum_trial_1,30C,30.0,aluminum,1,8.40066,50
1,30C__aluminum_trial_2,30C,30.0,aluminum,2,7.40070,50
2,30C__aluminum_trial_3,30C,30.0,aluminum,3,7.70070,51
3,30C__aluminum_trial_4,30C,30.0,aluminum,4,8.70070,51
4,30C__aluminum_trial_5,30C,30.0,aluminum,5,7.20070,51


# 4. Interpretable thermal-response features


In [6]:
def _smooth(values, window=11, polyorder=2):
    values = np.asarray(values, float)
    selected = min(window, len(values))
    if selected % 2 == 0:
        selected -= 1
    minimum = polyorder + 2
    if minimum % 2 == 0:
        minimum += 1
    return (
        savgol_filter(values, selected, polyorder, mode="interp")
        if selected >= minimum else values.copy()
    )


def _slope(time, values, window):
    mask = (time >= window[0]) & (time <= window[1])
    if mask.sum() < 3:
        return np.nan
    return float(np.polyfit(time[mask], values[mask], 1)[0])


def extract_thermal_features(trial):
    time = trial["time_from_contact"].to_numpy(float)
    primary = _smooth(trial["Primary"].to_numpy(float))
    secondary = _smooth(trial["Secondary"].to_numpy(float))
    difference = primary - secondary
    result = {}

    for name, signal in {
        "primary": primary,
        "secondary": secondary,
        "difference": difference,
    }.items():
        change = signal - signal[0]
        rate = np.gradient(signal, time)
        result[f"{name}_final_change"] = float(change[-1])
        result[f"{name}_max_abs_change"] = float(np.max(np.abs(change)))
        result[f"{name}_response_auc"] = float(np.trapezoid(np.abs(change), time))
        result[f"{name}_early_slope"] = _slope(time, signal, EARLY_WINDOW)
        result[f"{name}_mid_slope"] = _slope(time, signal, MID_WINDOW)
        result[f"{name}_late_slope"] = _slope(time, signal, LATE_WINDOW)
        result[f"{name}_max_abs_rate"] = float(np.max(np.abs(rate)))
        result[f"{name}_rate_auc"] = float(np.trapezoid(np.abs(rate), time))

    # Dimensionless/cross-sensor summaries.
    primary_auc = result["primary_response_auc"]
    result["secondary_primary_auc_ratio"] = (
        result["secondary_response_auc"] / primary_auc
        if not np.isclose(primary_auc, 0) else np.nan
    )
    result["initial_sensor_difference"] = float(difference[0])
    result["final_sensor_difference"] = float(difference[-1])
    return result


thermal_rows = []
for trial_id, trial in aligned_trials.items():
    features = extract_thermal_features(trial)
    features.update({
        "trial_id": trial_id,
        "Temperature": trial["Temperature"].iloc[0],
        "Temperature_C": float(trial["Temperature_C"].iloc[0]),
        "Sample": trial["Sample"].iloc[0],
        "Trial": int(trial["Trial"].iloc[0]),
        "k": float(trial["k"].iloc[0]),
        "eff": float(trial["eff"].iloc[0]),
    })
    thermal_rows.append(features)

THERMAL_FEATURES = pd.DataFrame(thermal_rows)
print(f"Thermal features: {len(THERMAL_FEATURES.columns) - 7}")
display(THERMAL_FEATURES.head())


Thermal features: 27


,primary_final_change,primary_max_abs_change,primary_response_auc,primary_early_slope,primary_mid_slope,primary_late_slope,primary_max_abs_rate,primary_rate_auc,secondary_final_change,secondary_max_abs_change,...,secondary_primary_auc_ratio,initial_sensor_difference,final_sensor_difference,trial_id,Temperature,Temperature_C,Sample,Trial,k,eff
0,-0.000003,0.000003,0.000013,-0.000002,-3.654261e-07,-1.344399e-07,0.000003,0.000003,-0.000002,0.000002,...,0.225534,-0.000040,-0.000042,30C__aluminum_trial_1,30C,30.0,aluminum,1,237.0,23958.094665
1,-0.000003,0.000003,0.000014,-0.000002,-3.595754e-07,-1.291998e-07,0.000004,0.000003,-0.000002,0.000002,...,0.241267,-0.000040,-0.000042,30C__aluminum_trial_2,30C,30.0,aluminum,2,237.0,23958.094665
2,-0.000003,0.000003,0.000014,-0.000002,-3.389437e-07,-1.112019e-07,0.000003,0.000003,-0.000002,0.000002,...,0.253393,-0.000040,-0.000042,30C__aluminum_trial_3,30C,30.0,aluminum,3,237.0,23958.094665
3,-0.000003,0.000003,0.000013,-0.000002,-3.724766e-07,-1.181206e-07,0.000003,0.000003,-0.000002,0.000002,...,0.243614,-0.000040,-0.000042,30C__aluminum_trial_4,30C,30.0,aluminum,4,237.0,23958.094665
4,-0.000003,0.000003,0.000013,-0.000002,-3.853054e-07,-1.244971e-07,0.000003,0.000003,-0.000002,0.000002,...,0.264204,-0.000041,-0.000042,30C__aluminum_trial_5,30C,30.0,aluminum,5,237.0,23958.094665


# 5. Manual ESN reservoir and compact trajectory summaries

Each temperature-specific trial becomes one row. Every reservoir unit contributes
nine summaries: mean, standard deviation, range, net change, slope, absolute area,
time of maximum absolute activation, skewness, and excess kurtosis.


In [7]:
class ManualReservoir:
    def __init__(
        self,
        res_size=30,
        leak_rate=0.9,
        input_magnitude=1.5,
        spectral_radius=1.3,
        washout=0,
        random_state=42,
    ):
        self.res_size = int(res_size)
        self.leak_rate = float(leak_rate)
        self.input_magnitude = float(input_magnitude)
        self.spectral_radius = float(spectral_radius)
        self.washout = int(washout)
        rng = np.random.default_rng(random_state)
        self.Win = (rng.random((self.res_size, 1 + 5)) - 0.5) * self.input_magnitude
        W = rng.random((self.res_size, self.res_size)) - 0.5
        radius = np.max(np.abs(linalg.eigvals(W)))
        if not np.isfinite(radius) or np.isclose(radius, 0):
            raise ValueError("Invalid reservoir spectral radius.")
        self.W = (W / radius.real) * self.spectral_radius

    def run(self, sequence):
        sequence = np.asarray(sequence, float)
        if sequence.ndim != 2 or sequence.shape[1] != 5:
            raise ValueError("Expected sequence with five input channels.")
        if len(sequence) <= self.washout:
            raise ValueError(
                f"Sequence has {len(sequence)} samples, but washout={self.washout}. "
                "Washout must be smaller than the sequence length."
            )
        x = np.zeros((self.res_size, 1))
        states = []
        for row in sequence:
            u = row.reshape(-1, 1)
            x = (
                (1 - self.leak_rate) * x
                + self.leak_rate * np.tanh(
                    self.Win @ np.vstack((1.0, u)) + self.W @ x
                )
            )
            states.append(x[:, 0].copy())
        return np.asarray(states)[self.washout:]


def raw_esn_input(trial):
    time = trial["time_from_contact"].to_numpy(float)
    primary = _smooth(trial["Primary"].to_numpy(float))
    secondary = _smooth(trial["Secondary"].to_numpy(float))
    difference = primary - secondary
    primary_rate = np.gradient(primary, time)
    secondary_rate = np.gradient(secondary, time)
    return np.column_stack([
        primary, secondary, difference, primary_rate, secondary_rate
    ])


def summarize_reservoir_trajectories(time, states):
    """Return finite statistical/dynamical summaries for every reservoir unit."""
    time = np.asarray(time, float)
    states = np.asarray(states, float)
    if states.ndim != 2 or len(time) != len(states):
        raise ValueError("Time and reservoir states must have matching rows.")
    if len(time) < 2 or np.any(~np.isfinite(time)) or np.any(np.diff(time) <= 0):
        raise ValueError("Reservoir-state time must be finite and strictly increasing.")
    if np.any(~np.isfinite(states)):
        raise ValueError("Reservoir states contain non-finite values.")

    result = {}
    for unit in range(states.shape[1]):
        values = states[:, unit]
        maximum_absolute_index = int(np.argmax(np.abs(values)))
        standard_deviation = float(np.std(values, ddof=0))

        # Constant or nearly constant trajectories have well-defined zero shape
        # rather than scipy's otherwise undefined skewness/kurtosis warnings.
        if len(values) < 3 or np.isclose(standard_deviation, 0.0):
            skewness = 0.0
        else:
            skewness = float(skew(values, bias=False))
        if len(values) < 4 or np.isclose(standard_deviation, 0.0):
            excess_kurtosis = 0.0
        else:
            excess_kurtosis = float(kurtosis(values, fisher=True, bias=False))

        summaries = {
            "mean": float(np.mean(values)),
            "std": standard_deviation,
            "range": float(np.ptp(values)),
            "net_change": float(values[-1] - values[0]),
            "slope": float(np.polyfit(time, values, 1)[0]),
            "absolute_area": float(np.trapezoid(np.abs(values), time)),
            "time_of_max_absolute": float(time[maximum_absolute_index]),
            "skewness": skewness,
            "excess_kurtosis": excess_kurtosis,
        }
        
        for name, value in summaries.items():
            if not np.isfinite(value):
                raise ValueError(f"Non-finite {name} for reservoir unit {unit}.")
            result[f"esn_{name}_u{unit:03d}"] = value
    return result


# 6. Leakage-safe metrics and XGBoost construction


In [8]:
def regression_metrics(y_true, y_pred):
    """Metrics requiring variation in y_true; intended for pooled or mixed-target data."""
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    if len(y_true) == 0 or np.any(~np.isfinite(y_true)) or np.any(~np.isfinite(y_pred)):
        raise ValueError("Metrics require non-empty, finite targets and predictions.")
    target_range = float(np.ptp(y_true))
    if len(y_true) < 2 or target_range <= 0:
        raise ValueError(
            "R² and test-range NRMSE require at least two distinct target values. "
            "Use constant_target_fold_metrics for a single-material LOMO fold."
        )
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    return {
        "n": len(y_true),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": rmse,
        "nrmse_range": rmse / target_range,
        "r2": float(r2_score(y_true, y_pred)),
        "median_ape_pct": float(np.median(np.abs((y_true - y_pred) / y_true)) * 100),
    }


def constant_target_fold_metrics(y_true, y_pred, training_target_range):
    """Valid diagnostics for one outer LOMO fold with a constant test target."""
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    training_target_range = float(training_target_range)
    if len(y_true) == 0 or np.any(~np.isfinite(y_true)) or np.any(~np.isfinite(y_pred)):
        raise ValueError("Fold metrics require non-empty, finite values.")
    if training_target_range <= 0:
        raise ValueError("The outer-training target range must be positive.")
    residual = y_pred - y_true
    rmse = float(np.sqrt(np.mean(residual ** 2)))
    return {
        "n": len(y_true),
        "mae": float(np.mean(np.abs(residual))),
        "rmse": rmse,
        "nrmse_training_range": rmse / training_target_range,
        "mean_error_bias": float(np.mean(residual)),
        "median_ape_pct": float(np.median(np.abs(residual / y_true)) * 100),
    }


def make_regressor(xgb_params=None, random_state=RANDOM_STATE):
    params = {**XGB_PARAMS, **(xgb_params or {})}
    xgb = XGBRegressor(
        objective="reg:squarederror",
        random_state=int(random_state),
        n_jobs=-1,
        tree_method="hist",
        verbosity=0,
        **params,
    )
    base = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("xgb", xgb),
    ])
    return TransformedTargetRegressor(
        regressor=base,
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    )


# 7. Fold-local multi-temperature ESN feature generation


In [9]:
def build_esn_feature_table(train_ids, all_ids, esn_params=None, random_state=RANDOM_STATE):
    params = {
        "res_size": ESN_RES_SIZE,
        "leak_rate": ESN_LEAK_RATE,
        "input_magnitude": ESN_INPUT_MAGNITUDE,
        "spectral_radius": ESN_SPECTRAL_RADIUS,
        "washout": ESN_WASHOUT,
    }
    params.update(esn_params or {})
    scaler = StandardScaler().fit(np.vstack([
        raw_esn_input(aligned_trials[trial_id]) for trial_id in train_ids
    ]))
    reservoir = ManualReservoir(**params, random_state=random_state)
    rows = []
    for trial_id in all_ids:
        trial = aligned_trials[trial_id]
        sequence = scaler.transform(raw_esn_input(trial))
        states = reservoir.run(sequence)
        state_time = trial["time_from_contact"].to_numpy(float)[params["washout"]:]
        features = summarize_reservoir_trajectories(state_time, states)
        features.update({
            "trial_id": trial_id,
            "Temperature": trial["Temperature"].iloc[0],
            "Temperature_C": float(trial["Temperature_C"].iloc[0]),
            "Sample": trial["Sample"].iloc[0],
            "Trial": int(trial["Trial"].iloc[0]),
            "k": float(trial["k"].iloc[0]),
            "eff": float(trial["eff"].iloc[0]),
        })
        rows.append(features)
    return pd.DataFrame(rows)

# 8. Unified Optuna TPE optimization with shared LOMO folds

For one Optuna trial, TPE nominates one joint ESN–XGBoost hyperparameter
combination. The notebook then runs all 14 LOMO folds with that exact same
combination. Each fold trains on 13 materials and predicts the remaining
material at all four temperatures.

The 14 held-out blocks are pooled into 336 out-of-fold predictions. Candidate
quality is primarily the pooled NRMSE; a smaller penalty is added when normalized
fold errors vary strongly across materials. Pooled R² remains a diagnostic rather
than a direct term in the objective.

TPE uses the first 20 completed trials as startup observations. It then models the
parameter values associated with relatively good and relatively poor objective
scores and nominates candidates that are more likely to fall in the good region.

Pruning is deliberately disabled. Every candidate must finish all 14 material
folds so its pooled objective is calculated from the same validation coverage.
MLflow logs one child run per Optuna trial under a single parent study run.


In [10]:
ESN_OPTUNA_SPACE = {
    "res_size": ("integer", 10, 80),
    "leak_rate": ("float", 0.05, 0.95),
    "input_magnitude": ("float", 0.25, 2.00),
    "spectral_radius": ("float", 0.30, 1.50),
    "washout": ("integer", 0, 5),
}

# The ranges include lightly and moderately regularized models. The previous upper
# regularization bounds allowed models that collapsed predictions toward the center.
XGB_OPTUNA_SPACE = {
    "n_estimators": ("integer", 200, 1200),
    "max_depth": ("integer", 2, 6),
    "learning_rate": ("log_float", 0.015, 0.20),
    "min_child_weight": ("log_float", 0.10, 10.0),
    "subsample": ("float", 0.65, 1.00),
    "colsample_bytree": ("float", 0.40, 1.00),
    "reg_alpha": ("log_float", 1e-6, 3.0),
    "reg_lambda": ("log_float", 0.01, 30.0),
}

OPTUNA_SPACE = {**ESN_OPTUNA_SPACE, **XGB_OPTUNA_SPACE}
ESN_OPTUNA_KEYS = tuple(ESN_OPTUNA_SPACE)
XGB_OPTUNA_KEYS = tuple(XGB_OPTUNA_SPACE)
OPTUNA_KEYS = tuple(OPTUNA_SPACE)


def _split_joint_parameters(params):
    missing = set(OPTUNA_KEYS) - set(params)
    if missing:
        raise KeyError(f"Joint candidate is missing parameters: {sorted(missing)}")
    esn_params = {key: params[key] for key in ESN_OPTUNA_KEYS}
    xgb_params = {key: params[key] for key in XGB_OPTUNA_KEYS}
    esn_params["res_size"] = int(esn_params["res_size"])
    esn_params["washout"] = int(esn_params["washout"])
    xgb_params["n_estimators"] = int(xgb_params["n_estimators"])
    xgb_params["max_depth"] = int(xgb_params["max_depth"])
    return esn_params, xgb_params


def _predict_joint_fold(train_meta, valid_meta, target, params, random_states):
    train_ids = train_meta["trial_id"].tolist()
    valid_ids = valid_meta["trial_id"].tolist()
    if set(train_ids) & set(valid_ids):
        raise RuntimeError("Trial leakage detected during Optuna search.")
    if set(train_meta["Sample"]) & set(valid_meta["Sample"]):
        raise RuntimeError("Material leakage detected during Optuna search.")

    esn_params, xgb_params = _split_joint_parameters(params)
    seed_predictions = []
    for random_state in random_states:
        features = build_esn_feature_table(
            train_ids,
            train_ids + valid_ids,
            esn_params=esn_params,
            random_state=random_state,
        ).set_index("trial_id")
        feature_cols = ["Temperature_C"] + [
            column for column in features if column.startswith("esn_")
        ]
        X_train = features.loc[train_ids, feature_cols].reset_index(drop=True)
        X_valid = features.loc[valid_ids, feature_cols].reset_index(drop=True)
        y_train = features.loc[train_ids, target].reset_index(drop=True)

        model = make_regressor(xgb_params=xgb_params, random_state=random_state)
        model.fit(X_train, y_train)
        seed_predictions.append(np.clip(
            model.predict(X_valid), y_train.min(), y_train.max()
        ))

    return valid_meta[target].to_numpy(float), np.mean(seed_predictions, axis=0)


def evaluate_unified_candidate(metadata, target, params, random_states, stability_weight):
    """Evaluate one shared candidate over all material-held-out folds."""
    splitter = LeaveOneGroupOut()
    fold_rows, prediction_rows = [], []
    target_range = float(np.ptp(metadata[target].to_numpy(float)))

    for fold, (train_idx, valid_idx) in enumerate(
        splitter.split(metadata, groups=metadata["Sample"]), start=1
    ):
        train_meta = metadata.iloc[train_idx].reset_index(drop=True)
        valid_meta = metadata.iloc[valid_idx].reset_index(drop=True)
        held_out = str(valid_meta["Sample"].iloc[0])
        y_true, y_pred = _predict_joint_fold(
            train_meta, valid_meta, target, params, tuple(random_states)
        )
        diagnostics = constant_target_fold_metrics(
            y_true, y_pred, np.ptp(train_meta[target].to_numpy(float))
        )
        fold_rows.append({
            "fold": fold,
            "held_out_material": held_out,
            **diagnostics,
        })
        prediction_rows.append(pd.DataFrame({
            "trial_id": valid_meta["trial_id"].to_numpy(),
            "Temperature": valid_meta["Temperature"].to_numpy(),
            "Temperature_C": valid_meta["Temperature_C"].to_numpy(),
            "Sample": valid_meta["Sample"].to_numpy(),
            "Trial": valid_meta["Trial"].to_numpy(),
            "fold": fold,
            "y_true": y_true,
            "y_pred": y_pred,
        }))

    folds = pd.DataFrame(fold_rows)
    predictions = pd.concat(prediction_rows, ignore_index=True)
    pooled = regression_metrics(predictions["y_true"], predictions["y_pred"])
    fold_nrmse = folds["nrmse_training_range"].to_numpy(float)
    prediction_range = float(np.ptp(predictions["y_pred"].to_numpy(float)))
    range_coverage = prediction_range / target_range if target_range > 0 else np.nan

    if not np.isfinite(pooled["nrmse_range"]) or not np.isfinite(pooled["r2"]):
        raise ValueError("Pooled candidate metrics are invalid.")
    if np.any(~np.isfinite(fold_nrmse)):
        raise ValueError("Fold-normalized candidate errors are invalid.")

    # Primary goal: overall pooled OOF accuracy. Secondary goal: similar error
    # behavior across held-out materials. R² and range coverage remain diagnostics.
    stability_sd = float(np.std(fold_nrmse, ddof=0))
    objective = float(pooled["nrmse_range"] + stability_weight * stability_sd)
    summary = {
        **pooled,
        "mean_fold_nrmse_training_range": float(np.mean(fold_nrmse)),
        "sd_fold_nrmse_training_range": stability_sd,
        "worst_fold_nrmse_training_range": float(np.max(fold_nrmse)),
        "prediction_range_coverage": range_coverage,
        "performance_index": objective,
    }
    return objective, folds, predictions, summary


def _suggest_joint_parameters(trial):
    # Ask Optuna TPE to nominate one complete ESN–XGBoost configuration.
    return {
        "res_size": trial.suggest_int("res_size", 10, 80),
        "leak_rate": trial.suggest_float("leak_rate", 0.05, 0.95),
        "input_magnitude": trial.suggest_float("input_magnitude", 0.25, 2.00),
        "spectral_radius": trial.suggest_float("spectral_radius", 0.30, 1.50),
        "washout": trial.suggest_int("washout", 0, 5),
        "n_estimators": trial.suggest_int("n_estimators", 200, 1200),
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.20, log=True),
        "min_child_weight": trial.suggest_float("min_child_weight", 0.10, 10.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.65, 1.00),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.40, 1.00),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-6, 3.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.01, 30.0, log=True),
    }


def _checkpoint_optuna_study(study, _trial):
    # Export completed Optuna records after every trial; not a resumable study DB.
    study.trials_dataframe(
        attrs=("number", "value", "params", "user_attrs", "state")
    ).to_csv(TRIAL_CHECKPOINT_FILE, index=False)


def optuna_optimize_unified(
    metadata,
    target="eff",
    n_trials=OPTUNA_N_TRIALS,
    n_startup_trials=OPTUNA_N_STARTUP_TRIALS,
    random_states=(42, 43, 44),
    stability_weight=STABILITY_WEIGHT,
    optimizer_seed=OPTUNA_SEED,
):
    # Optimize one shared configuration using multivariate Optuna TPE + MLflow.
    if n_trials < 2 or not 1 <= n_startup_trials <= n_trials:
        raise ValueError(
            "Require n_trials >= 2 and 1 <= n_startup_trials <= n_trials."
        )
    if metadata["Sample"].nunique() < 3:
        raise ValueError("At least three materials are required.")
    if not random_states:
        raise ValueError("At least one reservoir seed is required.")

    sampler = optuna.samplers.TPESampler(
        n_startup_trials=int(n_startup_trials),
        multivariate=bool(OPTUNA_MULTIVARIATE_TPE),
        seed=int(optimizer_seed),
    )
    study = optuna.create_study(
        study_name=f"{RESULT_STEM}_{MLFLOW_RUN_NAME}",
        direction="minimize",
        sampler=sampler,
    )

    # Evaluate known sensible configurations before TPE chooses all later trials.
    anchors = [
        {
            "res_size": 20, "leak_rate": 0.10, "input_magnitude": 1.0,
            "spectral_radius": 0.90, "washout": 0,
            "n_estimators": 500, "max_depth": 3, "learning_rate": 0.05,
            "min_child_weight": 1.0, "subsample": 0.85,
            "colsample_bytree": 0.85, "reg_alpha": 1e-4, "reg_lambda": 1.0,
        },
        {
            "res_size": 50, "leak_rate": 0.30, "input_magnitude": 1.0,
            "spectral_radius": 1.10, "washout": 0,
            "n_estimators": 800, "max_depth": 4, "learning_rate": 0.03,
            "min_child_weight": 0.5, "subsample": 0.90,
            "colsample_bytree": 0.90, "reg_alpha": 1e-5, "reg_lambda": 0.3,
        },
    ]
    for anchor in anchors:
        study.enqueue_trial(anchor)

    best_artifacts = {"loss": np.inf}
    records = []

    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow_client = mlflow.tracking.MlflowClient()
    experiment = mlflow_client.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
    if experiment is None:
        experiment_id = mlflow_client.create_experiment(
            MLFLOW_EXPERIMENT_NAME,
            artifact_location=MLFLOW_ARTIFACT_DIR.resolve().as_uri(),
        )
    else:
        experiment_id = experiment.experiment_id
    mlflow.set_experiment(experiment_id=experiment_id)

    with mlflow.start_run(run_name=MLFLOW_RUN_NAME) as parent_run:
        parent_run_id = parent_run.info.run_id
        mlflow.set_tags({
            "notebook": "4AAA",
            "optimizer": "optuna_multivariate_tpe",
            "model": "ESN_features_plus_XGBoost",
            "validation": "shared_hyperparameters_14_fold_LOMO",
            "target": target,
        })
        mlflow.log_params({
            "n_trials": int(n_trials),
            "n_startup_trials": int(n_startup_trials),
            "multivariate_tpe": bool(OPTUNA_MULTIVARIATE_TPE),
            "optimizer_seed": int(optimizer_seed),
            "stability_weight": float(stability_weight),
            "reservoir_seeds": ",".join(map(str, random_states)),
            "analysis_window": str(tuple(ANALYSIS_WINDOW)),
            "material_folds": int(metadata["Sample"].nunique()),
        })

        def objective(trial):
            params = _suggest_joint_parameters(trial)
            started = time.perf_counter()

            with mlflow.start_run(
                run_name=f"trial_{trial.number:03d}", nested=True
            ):
                mlflow.set_tags({
                    "optuna_trial_number": str(trial.number),
                    "acquisition": (
                        "startup_or_anchor"
                        if trial.number < n_startup_trials
                        else "tpe"
                    ),
                })
                mlflow.log_params(params)
                loss, folds, predictions, summary = evaluate_unified_candidate(
                    metadata,
                    target,
                    params,
                    tuple(random_states),
                    stability_weight,
                )
                elapsed = float(time.perf_counter() - started)

                metrics = {
                    "objective_J": float(loss),
                    "pooled_nrmse_range": float(summary["nrmse_range"]),
                    "pooled_r2": float(summary["r2"]),
                    "pooled_rmse": float(summary["rmse"]),
                    "pooled_mae": float(summary["mae"]),
                    "mean_fold_nrmse_training_range": float(
                        summary["mean_fold_nrmse_training_range"]
                    ),
                    "sd_fold_nrmse_training_range": float(
                        summary["sd_fold_nrmse_training_range"]
                    ),
                    "worst_fold_nrmse_training_range": float(
                        summary["worst_fold_nrmse_training_range"]
                    ),
                    "prediction_range_coverage": float(
                        summary["prediction_range_coverage"]
                    ),
                    "duration_seconds": elapsed,
                }
                mlflow.log_metrics(metrics)

                for _, fold_row in folds.iterrows():
                    material = str(fold_row["held_out_material"]).replace(" ", "_")
                    mlflow.log_metrics({
                        f"fold_{material}_rmse": float(fold_row["rmse"]),
                        f"fold_{material}_nrmse_training_range": float(
                            fold_row["nrmse_training_range"]
                        ),
                        f"fold_{material}_median_ape_pct": float(
                            fold_row["median_ape_pct"]
                        ),
                    })

                for key, value in metrics.items():
                    trial.set_user_attr(key, value)

            record = {
                "trial": int(trial.number + 1),
                "optuna_trial_number": int(trial.number),
                "acquisition": (
                    "startup_or_anchor"
                    if trial.number < n_startup_trials
                    else "tpe"
                ),
                **params,
                **metrics,
                "performance_index": float(loss),
            }
            records.append(record)

            if loss < best_artifacts["loss"]:
                best_artifacts.update({
                    "loss": float(loss),
                    "params": params.copy(),
                    "folds": folds.copy(),
                    "predictions": predictions.copy(),
                    "summary": summary.copy(),
                })

            print(
                f"Optuna trial {trial.number + 1:03d}/{n_trials}: "
                f"J={loss:.4f}, pooled NRMSE={summary['nrmse_range']:.4f}, "
                f"pooled R²={summary['r2']:.4f}, "
                f"elapsed={elapsed / 60:.1f} min"
            )
            return float(loss)

        study.optimize(
            objective,
            n_trials=int(n_trials),
            callbacks=[_checkpoint_optuna_study],
            gc_after_trial=True,
            show_progress_bar=False,
        )

        if "params" not in best_artifacts:
            raise RuntimeError("Optuna finished without a valid completed trial.")

        mlflow.log_metrics({
            "best_objective_J": float(best_artifacts["loss"]),
            "best_pooled_nrmse_range": float(
                best_artifacts["summary"]["nrmse_range"]
            ),
            "best_pooled_r2": float(best_artifacts["summary"]["r2"]),
        })
        mlflow.log_params({
            f"best_{key}": value for key, value in best_artifacts["params"].items()
        })
        if TRIAL_CHECKPOINT_FILE.exists():
            mlflow.log_artifact(str(TRIAL_CHECKPOINT_FILE), artifact_path="study")

    history = (
        pd.DataFrame(records)
        .sort_values("performance_index")
        .reset_index(drop=True)
    )
    best_params = best_artifacts["params"].copy()
    best_params["res_size"] = int(best_params["res_size"])
    best_params["washout"] = int(best_params["washout"])
    best_params["n_estimators"] = int(best_params["n_estimators"])
    best_params["max_depth"] = int(best_params["max_depth"])

    return (
        history,
        best_params,
        best_artifacts["folds"],
        best_artifacts["predictions"],
        best_artifacts["summary"],
        parent_run_id,
    )


In [11]:
OPTUNA_TARGET = OPTIMIZATION_TARGET
RANDOM_STATES = RESERVOIR_SEEDS
METADATA = THERMAL_FEATURES[
    ["trial_id", "Temperature", "Temperature_C", "Sample", "Trial", "k", "eff"]
].reset_index(drop=True)

(
    OPTUNA_SEARCH_HISTORY,
    RECOMMENDED_JOINT_PARAMETERS,
    UNIFIED_FOLD_METRICS,
    UNIFIED_OOF_PREDICTIONS,
    UNIFIED_SUMMARY,
    MLFLOW_PARENT_RUN_ID,
) = optuna_optimize_unified(
    METADATA,
    target=OPTUNA_TARGET,
    n_trials=OPTUNA_N_TRIALS,
    n_startup_trials=OPTUNA_N_STARTUP_TRIALS,
    random_states=RANDOM_STATES,
    stability_weight=STABILITY_WEIGHT,
    optimizer_seed=OPTUNA_SEED,
)

RECOMMENDED_ESN_PARAMETERS, RECOMMENDED_XGB_PARAMETERS = _split_joint_parameters(
    RECOMMENDED_JOINT_PARAMETERS
)
UNIFIED_POOLED_METRICS = pd.DataFrame([{
    "target": OPTUNA_TARGET,
    "protocol": "unified_shared_hyperparameters_grouped_lomo_optuna_tpe",
    "n_optuna_trials": OPTUNA_N_TRIALS,
    "random_states": tuple(RANDOM_STATES),
    **{key: UNIFIED_SUMMARY[key] for key in (
        "n", "mae", "rmse", "nrmse_range", "r2", "median_ape_pct"
    )},
    "prediction_range_coverage": UNIFIED_SUMMARY["prediction_range_coverage"],
}])
UNIFIED_FOLD_STABILITY = pd.DataFrame([{
    "folds": len(UNIFIED_FOLD_METRICS),
    "mean_fold_mae": UNIFIED_FOLD_METRICS["mae"].mean(),
    "sd_fold_mae": UNIFIED_FOLD_METRICS["mae"].std(ddof=1),
    "worst_fold_mae": UNIFIED_FOLD_METRICS["mae"].max(),
    "mean_fold_rmse": UNIFIED_FOLD_METRICS["rmse"].mean(),
    "sd_fold_rmse": UNIFIED_FOLD_METRICS["rmse"].std(ddof=1),
    "worst_fold_rmse": UNIFIED_FOLD_METRICS["rmse"].max(),
    "mean_fold_nrmse_training_range": UNIFIED_SUMMARY["mean_fold_nrmse_training_range"],
    "sd_fold_nrmse_training_range": UNIFIED_SUMMARY["sd_fold_nrmse_training_range"],
    "worst_fold_nrmse_training_range": UNIFIED_SUMMARY["worst_fold_nrmse_training_range"],
}])

temperature_rows = []
for temperature, part in UNIFIED_OOF_PREDICTIONS.groupby("Temperature", sort=True):
    temperature_rows.append({
        "Temperature": temperature,
        **regression_metrics(part["y_true"], part["y_pred"]),
    })
UNIFIED_TEMPERATURE_METRICS = pd.DataFrame(temperature_rows)

print("Selected Optuna-TPE shared-parameter grouped-LOMO performance:")
display(UNIFIED_POOLED_METRICS)
print("Per-material fold diagnostics (same hyperparameters in every fold):")
display(UNIFIED_FOLD_METRICS)
print("Across-material fold stability:")
display(UNIFIED_FOLD_STABILITY)
print("Pooled OOF performance by operating temperature:")
display(UNIFIED_TEMPERATURE_METRICS)
print("Recommended shared ESN parameters:")
display(pd.Series(RECOMMENDED_ESN_PARAMETERS))
print("Recommended shared XGBoost parameters:")
display(pd.Series(RECOMMENDED_XGB_PARAMETERS))
print("MLflow parent run ID:", MLFLOW_PARENT_RUN_ID)

SEARCH_HISTORY_FILE = RESULTS_DIR / f"{RESULT_STEM}_search_history.csv"
FOLD_METRICS_FILE = RESULTS_DIR / f"{RESULT_STEM}_fold_metrics.csv"
OOF_PREDICTIONS_FILE = RESULTS_DIR / f"{RESULT_STEM}_oof_predictions.csv"
POOLED_METRICS_FILE = RESULTS_DIR / f"{RESULT_STEM}_pooled_metrics.csv"
FOLD_STABILITY_FILE = RESULTS_DIR / f"{RESULT_STEM}_fold_stability.csv"
TEMPERATURE_METRICS_FILE = RESULTS_DIR / f"{RESULT_STEM}_temperature_metrics.csv"

OPTUNA_SEARCH_HISTORY.to_csv(SEARCH_HISTORY_FILE, index=False)
if UNIFIED_SUMMARY["r2"] <= 0:
    raise RuntimeError(
        "No acceptable unified configuration was found: best pooled OOF R² is "
        f"{UNIFIED_SUMMARY['r2']:.4f}. Increase OPTUNA_N_TRIALS or revise the "
        "search space; the parameter handoff file was not written."
    )

UNIFIED_FOLD_METRICS.to_csv(FOLD_METRICS_FILE, index=False)
UNIFIED_OOF_PREDICTIONS.to_csv(OOF_PREDICTIONS_FILE, index=False)
UNIFIED_POOLED_METRICS.to_csv(POOLED_METRICS_FILE, index=False)
UNIFIED_FOLD_STABILITY.to_csv(FOLD_STABILITY_FILE, index=False)
UNIFIED_TEMPERATURE_METRICS.to_csv(TEMPERATURE_METRICS_FILE, index=False)

payload = {
    "source": "one shared configuration selected by pooled grouped-LOMO Optuna TPE search",
    "temperature_folders": list(TEMPERATURE_FOLDERS),
    "target": OPTUNA_TARGET,
    "analysis_window": [float(value) for value in ANALYSIS_WINDOW],
    "thermal_windows": {
        "early": list(EARLY_WINDOW), "mid": list(MID_WINDOW), "late": list(LATE_WINDOW),
    },
    "summary_features": [
        "mean", "std", "range", "net_change", "slope", "absolute_area",
        "time_of_max_absolute", "skewness", "excess_kurtosis",
    ],
    "reservoir_seeds": [int(value) for value in RANDOM_STATES],
    "esn_parameters": {
        key: (int(value) if key in {"res_size", "washout"} else float(value))
        for key, value in RECOMMENDED_ESN_PARAMETERS.items()
    },
    "xgb_parameters": {
        key: (int(value) if key in {"n_estimators", "max_depth"} else float(value))
        for key, value in RECOMMENDED_XGB_PARAMETERS.items()
    },
    "selection_metrics": {
        "objective_J": float(UNIFIED_SUMMARY["performance_index"]),
        "pooled_nrmse_range": float(UNIFIED_SUMMARY["nrmse_range"]),
        "pooled_r2": float(UNIFIED_SUMMARY["r2"]),
        "prediction_range_coverage": float(UNIFIED_SUMMARY["prediction_range_coverage"]),
        "sd_fold_nrmse_training_range": float(
            UNIFIED_SUMMARY["sd_fold_nrmse_training_range"]
        ),
    },
    "optuna_search": {
        "sampler": "TPESampler",
        "multivariate": bool(OPTUNA_MULTIVARIATE_TPE),
        "n_trials": int(OPTUNA_N_TRIALS),
        "n_startup_trials": int(OPTUNA_N_STARTUP_TRIALS),
        "optimizer_seed": int(OPTUNA_SEED),
        "persistent_optuna_storage": False,
        "material_folds": int(METADATA["Sample"].nunique()),
        "stability_weight": float(STABILITY_WEIGHT),
        "search_space": {
            key: [kind, float(low), float(high)]
            for key, (kind, low, high) in OPTUNA_SPACE.items()
        },
    },
    "mlflow": {
        "tracking_uri": MLFLOW_TRACKING_URI,
        "backend": "local SQLite used only for MLflow tracking",
        "artifact_directory": str(MLFLOW_ARTIFACT_DIR.resolve()),
        "experiment_name": MLFLOW_EXPERIMENT_NAME,
        "parent_run_id": MLFLOW_PARENT_RUN_ID,
    },
}
with PARAMETER_FILE.open("w") as file:
    json.dump(payload, file, indent=2)

# Add final artifacts to the already-created MLflow parent run.
with mlflow.start_run(run_id=MLFLOW_PARENT_RUN_ID):
    for artifact in (
        SEARCH_HISTORY_FILE,
        FOLD_METRICS_FILE,
        OOF_PREDICTIONS_FILE,
        POOLED_METRICS_FILE,
        FOLD_STABILITY_FILE,
        TEMPERATURE_METRICS_FILE,
        PARAMETER_FILE,
    ):
        mlflow.log_artifact(str(artifact), artifact_path="best_trial_outputs")

print("Saved Optuna-TPE configuration for fixed analysis:", PARAMETER_FILE)
print("MLflow UI command:")
print(f'mlflow ui --backend-store-uri "{MLFLOW_TRACKING_URI}"')


/var/folders/fx/p7h99klx61g3f9j63ld4_xcm0000gn/T/ipykernel_27302/1847828992.py:180: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler = optuna.samplers.TPESampler(
2026/08/19 11:21:10 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/19 11:21:10 INFO mlflow.store.db.utils: Updating database tables


Optuna trial 001/100: J=0.2731, pooled NRMSE=0.2097, pooled R²=0.5168, elapsed=2.5 min
Optuna trial 002/100: J=0.2705, pooled NRMSE=0.2076, pooled R²=0.5264, elapsed=6.3 min
Optuna trial 003/100: J=0.2707, pooled NRMSE=0.2079, pooled R²=0.5248, elapsed=5.9 min
Optuna trial 004/100: J=0.2717, pooled NRMSE=0.2084, pooled R²=0.5229, elapsed=5.6 min
Optuna trial 005/100: J=0.2825, pooled NRMSE=0.2163, pooled R²=0.4857, elapsed=2.1 min
Optuna trial 006/100: J=0.2765, pooled NRMSE=0.2125, pooled R²=0.5038, elapsed=3.2 min
Optuna trial 007/100: J=0.2811, pooled NRMSE=0.2154, pooled R²=0.4901, elapsed=3.0 min
Optuna trial 008/100: J=0.2679, pooled NRMSE=0.2060, pooled R²=0.5336, elapsed=2.1 min
Optuna trial 009/100: J=0.2796, pooled NRMSE=0.2151, pooled R²=0.4914, elapsed=4.0 min
Optuna trial 010/100: J=0.2804, pooled NRMSE=0.2161, pooled R²=0.4866, elapsed=6.0 min
Optuna trial 011/100: J=0.2790, pooled NRMSE=0.2145, pooled R²=0.4943, elapsed=7.5 min
Optuna trial 012/100: J=0.2762, pooled NRMS

,target,protocol,n_optuna_trials,random_states,n,mae,rmse,nrmse_range,r2,median_ape_pct,prediction_range_coverage
0,eff,unified_shared_hyperparameters_grouped_lomo_op...,100,"(42, 43, 44)",336,3994.197198,7382.799923,0.198686,0.566119,49.107277,0.81115


Per-material fold diagnostics (same hyperparameters in every fold):


,fold,held_out_material,n,mae,rmse,nrmse_training_range,mean_error_bias,median_ape_pct
0,1,aluminum,24,6116.953454,7075.273152,0.190410,-5053.256465,21.346002
1,2,bismuth,24,6217.508823,6916.652021,0.186141,5801.957947,220.552937
2,3,cement,24,1219.337891,2532.616471,0.068158,1184.383213,14.130698
3,4,copper,24,23021.326553,23397.200529,0.977997,-23021.326553,58.772181
4,5,cork,24,133.221258,168.455446,0.004533,87.030726,69.942704
5,6,graphite,24,3485.088795,4375.296919,0.117748,-465.649881,27.683420
6,7,gypsum,24,609.690931,1030.802662,0.027741,494.307942,62.107472
7,8,iron,24,4280.218207,5475.483657,0.147356,3570.483505,17.736434
8,9,nickel,24,4351.650283,4966.638922,0.133662,-3332.350635,20.871635
9,10,pdms,24,195.418180,226.858109,0.006105,-195.418180,37.063984


...

Recommended shared ESN parameters:


res_size           22.000000
leak_rate           0.361991
input_magnitude     1.583048
spectral_radius     1.160180
washout             3.000000
dtype: float64

Recommended shared XGBoost parameters:


n_estimators        335.000000
max_depth             5.000000
learning_rate         0.097712
min_child_weight      0.157276
subsample             0.719668
colsample_bytree      0.495283
reg_alpha             0.000002
reg_lambda            0.564496
dtype: float64

MLflow parent run ID: ca00cfe29cdf49648058dd019e2e485c
Saved Optuna-TPE configuration for fixed analysis: /Users/markkeanujamesexconde/Library/CloudStorage/OneDrive-Personal/Documents/Academe Files/4. Doctoral/Research 1 - Warmth Sensor/Warmth-Sensor-Object-Detection/results/multi_temp_esn/30C-40C-50C-60C_eff_v7_optuna_tpe_mlflow_parameters.json
MLflow UI command:
mlflow ui --backend-store-uri "sqlite:////Users/markkeanujamesexconde/Library/CloudStorage/OneDrive-Personal/Documents/Academe Files/4. Doctoral/Research 1 - Warmth Sensor/Warmth-Sensor-Object-Detection/results/multi_temp_esn/4AAA_mlflow_tracking.db"


## Reading 4AAA outputs

- There is one optimized ESN–XGBoost configuration, not one configuration per material.
- Every Optuna candidate is tested across all 14 material-held-out folds.
- `pooled_metrics` summarizes the 336 pooled out-of-fold predictions.
- `fold_metrics` identifies difficult held-out materials using the same HPs.
- `search_history` ranks the 100 completed TPE candidates by objective `J`.
- MLflow stores the study as one parent run and each candidate as a child run.
- The CSV checkpoint preserves completed results, but without persistent Optuna
  storage the TPE sampler state cannot resume after a kernel interruption.
- MLflow uses a separate local SQLite backend only for experiment tracking.
- Because the same LOMO folds select the hyperparameters, the reported values are
  model-selection estimates rather than a separate untouched final test estimate.


# 9. Handoff

Run 4AAA when the pooled data, analysis windows, ESN summaries, Optuna search space,
or XGBoost search space changes. The notebook writes a `v7_optuna_tpe_mlflow`
parameter JSON with the same core parameter schema used by 4AA.

Before using the result in 4BB, point 4BB's parameter-file configuration to the
new `v7_optuna_tpe_mlflow_parameters.json` file. Keep the resulting fixed-parameter
evaluation separate from MLflow's hyperparameter-selection metrics.
